# Number of Children Under Age 1 in Michigan

Calculates the weighted count of persons with `age < 1` in Michigan.

**Two approaches compared:**
1. State-specific dataset (`MI.h5`)
2. National dataset filtered to Michigan via `state_code`

**Result at filing (2026-05-19, issue #135):** the state dataset undercounted MI population by ~59% (4.10M vs ~10.1M actual; 43,689 under age 1). See the status note below — that defect was fixed upstream the next day, and the committed outputs now reflect a 2026-07-23 rerun, which surfaces a *different* under-1 gap in the national dataset ([populace#520](https://github.com/PolicyEngine/populace/issues/520)).

## Status (2026-07-23 rerun, takeover review)

- **The #135 undercount is fixed upstream.** `states/MI.h5` was replaced on Hugging Face in the policyengine-us-data `1.115.4-patch` promotion (2026-05-20 — the day after #135 was filed). Rerunning with policyengine-us 1.782.0: MI total is now **10,119,131** (≈ Census ~10.1M), vs **4,104,577** at filing.
- **A different defect surfaced in the national default.** The national dataset is now the pinned populace release (`populace_us_2024…20260619T002242Z`); it puts MI under-1 at **40,204** vs ~104,000 annual MDHHS births — and gives CA 21.7% of all US infants vs 11.6% of US population. Tracked in [populace#520](https://github.com/PolicyEngine/populace/issues/520).
- The state file's under-1 count (**61,938**) also remains well below the births anchor: the two artifacts disagree with each other *and* with vital statistics on infants, even though totals now agree with Census.
- The as-filed outputs are preserved in [#135](https://github.com/PolicyEngine/analysis-notebooks/issues/135) and PR [#136](https://github.com/PolicyEngine/analysis-notebooks/pull/136).

In [1]:
from policyengine_us import Microsimulation
from huggingface_hub import hf_hub_download

YEAR = 2026

## Approach 1: State-specific MI.h5 dataset

In [2]:
mi_path = hf_hub_download(
    repo_id="policyengine/policyengine-us-data",
    filename="states/MI.h5",
    repo_type="model",
)
sim_state = Microsimulation(dataset=mi_path)
age_state = sim_state.calculate("age", YEAR)

# (boolean_series).sum() applies weights once -> weighted count of True rows
state_total = (age_state >= 0).sum()
state_under_1 = (age_state < 1).sum()
print(f"MI total population (state dataset): {state_total:,.0f}")
print(f"MI children under age 1 (state dataset): {state_under_1:,.0f}")

MI total population (state dataset): 10,119,131
MI children under age 1 (state dataset): 61,938


## Approach 2: National dataset filtered to Michigan

In [3]:
sim_nat = Microsimulation()
age_nat = sim_nat.calculate("age", YEAR).values
weight_nat = sim_nat.calculate("person_weight", YEAR).values
state_code_person = sim_nat.calculate("state_code", YEAR, map_to="person").values

in_mi = state_code_person == "MI"
mi_total = float(weight_nat[in_mi].sum())
mi_under_1 = float(weight_nat[(age_nat < 1) & in_mi].sum())
print(f"MI total population (national dataset): {mi_total:,.0f}")
print(f"MI children under age 1 (national dataset): {mi_under_1:,.0f}")

MI total population (national dataset): 10,307,213
MI children under age 1 (national dataset): 40,204


## Comparison

In [4]:
print(f"{'Source':<35} {'MI total':>15} {'Age < 1':>12}")
print("-" * 64)
print(f"{'State MI.h5 dataset':<35} {state_total:>15,.0f} {state_under_1:>12,.0f}")
print(f"{'National dataset (filter to MI)':<35} {mi_total:>15,.0f} {mi_under_1:>12,.0f}")
print(f"{'Census / MDHHS (real-world)':<35} {'~10,100,000':>15} {'~104,000':>12}")

Source                                     MI total      Age < 1
----------------------------------------------------------------
State MI.h5 dataset                      10,119,131       61,938
National dataset (filter to MI)          10,307,213       40,204
Census / MDHHS (real-world)             ~10,100,000     ~104,000
